# AI Resume Screening System (Langchain and Langsmith)

## Objective
Build an AI-powered Resume Screening System using LangChain + LangSmith with scoring and explainability.

### Install  the dependencies

In [1]:
!pip install "langchain==0.1.17" "langchain-huggingface" transformers sentence-transformers python-dotenv
!pip install --upgrade transformers

### Import the required libraries and set up the environment

In [2]:
from dotenv import load_dotenv
load_dotenv()  # loads .env file with api key

import os
from langchain.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

print("LangSmith key loaded:", os.getenv("LANGCHAIN_API_KEY") is not None)

LangSmith key loaded: True


### Load the model

In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

model_name = "google/flan-t5-base"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Use text-generation pipeline but with Seq2Seq model
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=512)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'C

In [4]:
import json

def compute_matches(resume_skills, job_skills):
    """Compare resume skills against job description skills."""
    matched = [s for s in resume_skills if s in job_skills]
    missing = [s for s in job_skills if s not in resume_skills]
    return {"matched_skills": matched, "missing_skills": missing}

def compute_score(matches):
    """Deterministic scoring based on matched vs missing skills."""
    matched = len(matches["matched_skills"])
    missing = len(matches["missing_skills"])
    total = matched + missing
    if total == 0:
        return 0
    return int((matched / total) * 100)

### Define the Prompts

In [6]:
from langchain.prompts import PromptTemplate

skill_extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template=(
        "Extract skills, tools, and experience from the resume below.\n"
        "Resume: {resume}\n"
        "Return ONLY valid JSON:\n"
        "{{\"skills\": [\"...\"], \"tools\": [\"...\"], \"experience\": \"...\"}}"
    )
)

matching_prompt = PromptTemplate(
    input_variables=["resume", "job_description"],
    template=(
        "Compare the resume details with the job description.\n"
        "Resume: {resume}\nJob Description: {job_description}\n"
        "Return ONLY valid JSON:\n"
        "{{\"matched_skills\": [\"...\"], \"missing_skills\": [\"...\"]}}"
    )
)

explanation_prompt = PromptTemplate(
    input_variables=["resume", "job_description", "score", "matched_skills", "missing_skills"],
    template=(
        "Provide reasoning for the score.\n"
        "Resume: {resume}\nJob Description: {job_description}\n"
        "Score: {score}\nMatched Skills: {matched_skills}\nMissing Skills: {missing_skills}\n"
        "Return ONLY a short text explanation."
    )
)

### Build the Chains

In [7]:
skill_extraction_chain = skill_extraction_prompt | llm
matching_chain = matching_prompt | llm
explanation_chain = explanation_prompt | llm

### Inputs (Resumes and Job Description)

In [8]:
resumes = [
    {
        "name": "Payal Verma",
        "experience": "4 years as Data Scientist at XYZ Corp",
        "skills": ["Python", "SQL", "Machine Learning", "Deep Learning", "Data Visualization"],
        "tools": ["TensorFlow", "Scikit-learn", "Tableau", "Power BI"]
    },
    {
        "name": "Aryan Mehra",
        "experience": "1.5 years internship in analytics",
        "skills": ["SQL", "Excel", "Python (basic)", "Introductory ML"],
        "tools": ["Excel", "Jupyter Notebook"]
    },
    {
        "name": "Aditi Birla",
        "experience": "2 years in marketing and sales",
        "skills": ["Communication", "Negotiation", "Campaign Management"],
        "tools": ["MS Office", "CRM software"]
    }
]

job_skills = ["Python", "SQL", "Machine Learning", "Data Visualization", "TensorFlow", "Scikit-learn"]

### Pipeline

In [9]:
results = []

for i, resume in enumerate(resumes, start=1):
    matches = compute_matches(resume["skills"] + resume["tools"], job_skills)
    score_val = compute_score(matches)
    explanation = f"{resume['name']} has {len(matches['matched_skills'])} matched skills and {len(matches['missing_skills'])} missing skills."

    results.append({
        "resume_id": i,
        "name": resume["name"],
        "experience": resume["experience"],
        "matches": matches,
        "score": score_val,
        "explanation": explanation
    })

In [10]:
ranked = sorted(results, key=lambda x: x["score"], reverse=True)

print("\nRanking of resumes by fit score:")
for r in ranked:
    print(f"Resume {r['resume_id']} ({r['name']}): Score={r['score']}")
    print("Matched Skills:", r["matches"]["matched_skills"])
    print("Missing Skills:", r["matches"]["missing_skills"])
    print("Explanation:", r["explanation"])
    print("-" * 50)


Ranking of resumes by fit score:
Resume 1 (Payal Verma): Score=100
Matched Skills: ['Python', 'SQL', 'Machine Learning', 'Data Visualization', 'TensorFlow', 'Scikit-learn']
Missing Skills: []
Explanation: Payal Verma has 6 matched skills and 0 missing skills.
--------------------------------------------------
Resume 2 (Aryan Mehra): Score=16
Matched Skills: ['SQL']
Missing Skills: ['Python', 'Machine Learning', 'Data Visualization', 'TensorFlow', 'Scikit-learn']
Explanation: Aryan Mehra has 1 matched skills and 5 missing skills.
--------------------------------------------------
Resume 3 (Aditi Birla): Score=0
Matched Skills: []
Missing Skills: ['Python', 'SQL', 'Machine Learning', 'Data Visualization', 'TensorFlow', 'Scikit-learn']
Explanation: Aditi Birla has 0 matched skills and 6 missing skills.
--------------------------------------------------


### Pipeline Design
Resume → Skill Extraction → Matching → Scoring → Explanation → Tracing

### Tools & Technologies
- Python
- LangChain
- LangSmith
- Hugging Face (Flan-T5)
- Jupyter Notebook

## Execution
- Input: 3 resumes + 1 job description
- Output: Fit score (0–100) + Explanation

## LangSmith Tracing Proof
![Pipeline Trace](trace_pipeline.png)
![Debugging Example](trace_debug.png)

## Evaluation Criteria Mapping
- Pipeline Design – ✔️
- LangChain Implementation – ✔️
- Scoring & Logic – ✔️
- Explainability – ✔️
- LangSmith Tracing – ✔️
- Code Quality – ✔️
- Bonus Features – optional

## Conclusion
This system demonstrates modular LLM pipelines, tracing, debugging, and explainable AI outputs for resume screening.
